In [4]:
import requests
BASE_URL = "https://api-web.nhle.com/v1"

In [2]:
url = f"{BASE_URL}/standings/now"
resp = requests.get(url, timeout=15)
resp.raise_for_status()
data = resp.json()
data

{'wildCardIndicator': True,
 'standingsDateTimeUtc': '2026-09-05T03:47:15Z',
 'standings': [{'clinchIndicator': 'p',
   'conferenceAbbrev': 'W',
   'conferenceHomeSequence': 1,
   'conferenceL10Sequence': 2,
   'conferenceName': 'Western',
   'conferenceRoadSequence': 1,
   'conferenceSequence': 1,
   'date': '2026-04-17',
   'divisionAbbrev': 'C',
   'divisionHomeSequence': 1,
   'divisionL10Sequence': 1,
   'divisionName': 'Central',
   'divisionRoadSequence': 1,
   'divisionSequence': 1,
   'gameTypeId': 2,
   'gamesPlayed': 82,
   'goalDifferential': 99,
   'goalDifferentialPctg': 1.207317,
   'goalAgainst': 203,
   'goalFor': 302,
   'goalsForPctg': 3.682927,
   'homeGamesPlayed': 41,
   'homeGoalDifferential': 49,
   'homeGoalsAgainst': 108,
   'homeGoalsFor': 157,
   'homeLosses': 9,
   'homeOtLosses': 6,
   'homePoints': 58,
   'homeRegulationPlusOtWins': 25,
   'homeRegulationWins': 25,
   'homeTies': 0,
   'homeWins': 26,
   'l10GamesPlayed': 10,
   'l10GoalDifferential': 14,

In [2]:
import pymysql

conn = pymysql.connect(
    host='localhost',
    user='root',
    password='1234',
    database='nhl_teams'
)
cursor = conn.cursor()

# Teams

In [4]:
cursor.execute("DROP TABLE IF EXISTS teams")
cursor.execute("""CREATE TABLE teams (
            id INT AUTO_INCREMENT PRIMARY KEY,
            abbrev VARCHAR(10),
            name VARCHAR(100),
            conference VARCHAR(100),
            division VARCHAR(100),
            logo_url TEXT
                )""")

for team in data['standings']:
    values = (
        team['teamAbbrev']['default'],
        team['teamName']['default'],
        team['conferenceName'],
        team['divisionName'],
        team['teamLogo']
    )

    insert_query = """INSERT INTO teams (abbrev, name, conference, division, logo_url) 
              VALUES (%s, %s, %s, %s, %s)"""

    cursor.execute(insert_query, values)

   
conn.commit()

  # Standings

In [6]:
cursor.execute("DROP TABLE IF EXISTS standings")

# 1. Create table ONCE, before the loop
cursor.execute("""CREATE TABLE IF NOT EXISTS standings (
            standing_id INT AUTO_INCREMENT PRIMARY KEY,
            team_id INT,
            season_id VARCHAR(20),
            games_played INT,
            wins INT,
            losses INT,
            ot_losses INT,
            points INT,
            goals_for INT,
            goals_against INT,
            home_wins INT,
            away_wins INT,
            streak_type VARCHAR(20),
            streak_count INT,
            FOREIGN KEY (team_id) REFERENCES teams(id)
            )""")

for team in data['standings']:
    abbrev = team['teamAbbrev']['default']

    # 2. Look up the team_id you already inserted into `teams`
    cursor.execute("SELECT id FROM teams WHERE abbrev = %s", (abbrev,))
    team_id = cursor.fetchone()[0]

    # 3. Use the REAL field names from the API
    values2 = (
        team_id,
        team['seasonId'],
        team['gamesPlayed'],
        team['wins'],
        team['losses'],
        team['otLosses'],
        team['points'],
        team.get('goalFor'),
        team.get('goalAgainst'),
        team.get('homeWins'),
        team.get('roadWins'),
        team.get('streakCode'),
        team.get('streakCount')
    )

    insert_query2 = """INSERT INTO standings 
        (team_id, season_id, games_played, wins, losses, ot_losses, points, 
         goals_for, goals_against, home_wins, away_wins, streak_type, streak_count)
        VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""
    cursor.execute(insert_query2, values2)
conn.commit()

# Players

In [7]:
cursor.execute("DROP TABLE IF EXISTS players")
cursor.execute("""CREATE TABLE players (
                    player_id BIGINT PRIMARY KEY,
                    team_id INT,
                    first_name VARCHAR(100),
                    last_name VARCHAR(100),
                    position VARCHAR(10),
                    jersey_number INT,
                    birth_date DATE,
                    birth_country VARCHAR(10),
                    height_cm REAL,
                    weight_kg REAL,
                    shoots_catches VARCHAR(5),
                    headshot_url TEXT,
                    FOREIGN KEY (team_id) REFERENCES teams(id)
                )""")

# Get every team abbrev + id we already loaded into `teams`
cursor.execute("SELECT id, abbrev FROM teams")
team_lookup = {abbrev: team_id for team_id, abbrev in cursor.fetchall()}

for abbrev, team_id in team_lookup.items():
    url = f"https://api-web.nhle.com/v1/roster/{abbrev}/current"
    response = requests.get(url)
    data = response.json()

    for group in ("forwards", "defensemen", "goalies"):
        for player in data.get(group, []):
            values = (
                player['id'],
                team_id,
                player['firstName']['default'],
                player['lastName']['default'],
                player.get('positionCode'),
                player.get('sweaterNumber'),
                player.get('birthDate'),
                player.get('birthCountry'),
                player.get('heightInCentimeters'),
                player.get('weightInKilograms'),
                player.get('shootsCatches'),
                player.get('headshot')
            )

            insert_query = """INSERT INTO players 
                (player_id, team_id, first_name, last_name, position, jersey_number,
                 birth_date, birth_country, height_cm, weight_kg, shoots_catches, headshot_url)
                VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""

            cursor.execute(insert_query, values)

conn.commit()

# Games

In [8]:
cursor.execute("DROP TABLE IF EXISTS games")
cursor.execute("""CREATE TABLE games (
                    game_id BIGINT PRIMARY KEY,
                    season VARCHAR(20),
                    game_type INT,
                    game_date DATE,
                    home_team_id INT,
                    away_team_id INT,
                    home_score INT,
                    away_score INT,
                    game_state VARCHAR(20),
                    venue_name VARCHAR(150),
                    FOREIGN KEY (home_team_id) REFERENCES teams(id),
                    FOREIGN KEY (away_team_id) REFERENCES teams(id)
                ) """)

cursor.execute("SELECT id, abbrev FROM teams")
team_lookup = {abbrev: team_id for team_id, abbrev in cursor.fetchall()}

for abbrev in team_lookup:
    url = f"https://api-web.nhle.com/v1/club-schedule-season/{abbrev}/now"
    response = requests.get(url)
    data = response.json()

    for game in data.get('games', []):
        values = (
            game['id'],
            str(game.get('season')),
            game.get('gameType'),
            game.get('gameDate'),
            team_lookup[game['homeTeam']['abbrev']],
            team_lookup[game['awayTeam']['abbrev']],
            game['homeTeam'].get('score'),
            game['awayTeam'].get('score'),
            game.get('gameState'),
            game.get('venue', {}).get('default')
        )

        insert_query = """INSERT IGNORE INTO games 
            (game_id, season, game_type, game_date, home_team_id, away_team_id,
             home_score, away_score, game_state, venue_name)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""

        cursor.execute(insert_query, values)

conn.commit()
  

# GAME STATS

In [ ]:
cursor.execute("DROP TABLE IF EXISTS game_stats")
cursor.execute("""CREATE TABLE game_stats (
    stat_id INT AUTO_INCREMENT PRIMARY KEY,
    game_id BIGINT,
    player_id BIGINT,
    team_id INT,
    goals INT,
    assists INT,
    points INT,
    shots_on_goal INT,
    penalty_min INT,
    toi VARCHAR(10),
    plus_minus INT,
    FOREIGN KEY (game_id) REFERENCES games(game_id),
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id) REFERENCES teams(id)
) """)

cursor.execute("SELECT id, abbrev FROM teams")
team_lookup = {abbrev: team_id for team_id, abbrev in cursor.fetchall()}

cursor.execute("SELECT game_id FROM games")
all_game_ids = [row[0] for row in cursor.fetchall()]

insert_query = """INSERT IGNORE INTO game_stats
    (game_id, player_id, team_id, goals, assists, points,
     shots_on_goal, penalty_min, toi, plus_minus)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""

for game_id in all_game_ids:
    url = f"https://api-web.nhle.com/v1/gamecenter/{game_id}/boxscore"
    response = requests.get(url)
    data = response.json()

    for team_side in ("homeTeam", "awayTeam"):
        team_abbrev = data[team_side]["abbrev"]
        team_id = team_lookup.get(team_abbrev)

        side_stats = data.get("playerByGameStats", {}).get(team_side)
        skaters = side_stats["forwards"] + side_stats["defense"]

        for player in skaters:
            values = (
                game_id,
                player['playerId'],
                team_id,
                player.get('goals', 0),
                player.get('assists', 0),
                player.get('points', 0),
                player.get('shots', 0),
                player.get('pim', 0),
                player.get('toi'),
                player.get('plusMinus', 0)
            )
            cursor.execute(insert_query, values)

conn.commit()

# Skater season stats

In [14]:
cursor.execute("DROP TABLE IF EXISTS skater_season_stats")
cursor.execute("""CREATE TABLE skater_season_stats (
    stat_id INT AUTO_INCREMENT PRIMARY KEY,
    player_id BIGINT,
    season VARCHAR(20),
    team_id INT,
    games_played INT,
    goals INT,
    assists INT,
    points INT,
    plus_minus INT,
    penalty_min INT,
    shots INT,
    avg_toi VARCHAR(10),
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id) REFERENCES teams(id)
) ENGINE=InnoDB""")

cursor.execute("DROP TABLE IF EXISTS goalie_season_stats")
cursor.execute("""CREATE TABLE goalie_season_stats (
    stat_id INT AUTO_INCREMENT PRIMARY KEY,
    player_id BIGINT,
    season VARCHAR(20),
    team_id INT,
    games_played INT,
    wins INT,
    losses INT,
    ot_losses INT,
    save_pct FLOAT,
    goals_against_avg FLOAT,
    shutouts INT,
    saves INT,
    FOREIGN KEY (player_id) REFERENCES players(player_id),
    FOREIGN KEY (team_id) REFERENCES teams(id)
) ENGINE=InnoDB""")

cursor.execute("SELECT id, abbrev FROM teams")
team_lookup = {abbrev: team_id for team_id, abbrev in cursor.fetchall()}

cursor.execute("SELECT player_id FROM players")
all_player_ids = [row[0] for row in cursor.fetchall()]

skater_query = """INSERT IGNORE INTO skater_season_stats
    (player_id, season, team_id, games_played, goals, assists, points,
     plus_minus, penalty_min, shots, avg_toi)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""

goalie_query = """INSERT IGNORE INTO goalie_season_stats
    (player_id, season, team_id, games_played, wins, losses, ot_losses,
     save_pct, goals_against_avg, shutouts, saves)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s)"""

for player_id in all_player_ids:
    url = f"https://api-web.nhle.com/v1/player/{player_id}/landing"
    response = requests.get(url)
    data = response.json()

    season_block = data.get("featuredStats", {}).get("regularSeason", {}).get("subSeason", {})
    season = data.get("featuredStats", {}).get("season")
    team_id = team_lookup.get(data.get("currentTeamAbbrev"))

    if data.get("position") == "G":
        values = (
            player_id, str(season), team_id,
            season_block.get("gamesPlayed"),
            season_block.get("wins"),
            season_block.get("losses"),
            season_block.get("otLosses"),
            season_block.get("savePctg"),
            season_block.get("goalsAgainstAverage"),
            season_block.get("shutouts"),
            season_block.get("saves")
        )
        cursor.execute(goalie_query, values)
    else:
        values = (
            player_id, str(season), team_id,
            season_block.get("gamesPlayed"),
            season_block.get("goals"),
            season_block.get("assists"),
            season_block.get("points"),
            season_block.get("plusMinus"),
            season_block.get("pim"),
            season_block.get("shots"),
            season_block.get("avgToi")
        )
        cursor.execute(skater_query, values)

conn.commit()